<a href="https://colab.research.google.com/github/nessa1234/NessaK/blob/master/AI-powered%20paraphrasing%20tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jre-headless

!pip install -q \
    transformers \
    sentencepiece \
    torch \
    sentence-transformers \
    language-tool-python \
    rouge-score \
    nltk

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jre-headless_17.0.20.1+1-1~24.04_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.20.1+1-1~24.04) ...
Setting up openjdk-17-jre-headless:amd64 (17.0.20.1+1-1~24.04) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.027s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.027s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
done.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━

In [2]:
import re
import torch
import nltk
import language_tool_python

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util

from nltk.tokenize import sent_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

from rouge_score import rouge_scorer
from difflib import SequenceMatcher

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cpu


In [4]:
MODEL_NAME = "humarin/chatgpt_paraphraser_on_T5_base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

print("Paraphrasing model loaded successfully!")

config.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Paraphrasing model loaded successfully!


In [5]:
similarity_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Semantic similarity model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic similarity model loaded successfully!


In [6]:
grammar_tool = language_tool_python.LanguageTool("en-US")

print("Grammar and spelling checker loaded successfully!")

Grammar and spelling checker loaded successfully!


In [21]:
def paraphrase_sentence(sentence):
    """
    Generate a paraphrase for a single sentence
    using a T5 Transformer model.
    """

    text = "paraphrase: " + sentence

    encoding = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,

            max_length=256,

            # Standard Beam Search
            num_beams=5,

            # We only need one final paraphrase
            num_return_sequences=1,

            # Avoid repeated phrases
            repetition_penalty=2.0,
            no_repeat_ngram_size=2,

            early_stopping=True
        )

    paraphrase = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return paraphrase

In [32]:
test_sentence = "Artificial intelligence is changing the way businesses operate."

result = paraphrase_sentence(test_sentence)

print("Original:")
print(test_sentence)

print("\nParaphrased:")
print(result)

Original:
Artificial intelligence is changing the way businesses operate.

Paraphrased:
The implementation of artificial intelligence is altering the business landscape.


In [33]:
print("=" * 60)
print("       AI-POWERED PARAPHRASING TOOL")
print("=" * 60)

user_text = input(
    "\nEnter the text you want to paraphrase:\n\n"
)

final_text, evaluation = ai_paraphraser(user_text)

       AI-POWERED PARAPHRASING TOOL

Enter the text you want to paraphrase:

Artificial intelligence is changing the way businesses operate. Companies are using machine learning to analyse large amounts of data and make better decisions. These technologies can improve efficiency and reduce the amount of repetitive work performed by employees.

Original Text:
------------------------------------------------------------
Artificial intelligence is changing the way businesses operate. Companies are using machine learning to analyse large amounts of data and make better decisions. These technologies can improve efficiency and reduce the amount of repetitive work performed by employees.

AI Generated Paraphrase:
------------------------------------------------------------
The implementation of artificial intelligence is altering the business landscape. Companies are leveraging machine learning to analyze vast amounts of data and improve their decisions. The implementation of these technologi